# Laboratorio 10.1 — Fundamentos de LangGraph

Construiremos un StateGraph pequeño desde sus nodos y bordes para hacer visibles el estado, el routing condicional, la persistencia y la pausa de aprobación humana.

## 1. Objetivos de aprendizaje

Al finalizar podrás:
- Definir un estado tipado compartido entre nodos.
- Crear nodos puros y conectar bordes y rutas condicionales.
- Compilar e invocar un grafo LangGraph.
- Reanudar una ejecución persistida después de una pausa para revisión humana.

## 2. Conceptos

LangGraph representa un workflow como estado y nodos que reciben una instantánea y devuelven cambios. Los bordes definen el orden; los condicionales eligen el siguiente nodo. Un checkpointer guarda estado por `thread_id`. Una interrupción permite pausar antes de una acción y reanudarla después de revisión; la aprobación humana debe ocurrir en la aplicación que presenta esa pausa.

## 3. Arquitectura

```mermaid
flowchart LR
    A[START] --> B[clasificador]
    B -->|tool| C[herramienta]
    B -->|direct| D[respuesta directa]
    C --> E[validacion Pydantic]
    D --> E
    E --> F[END]
    C -. interrupt_before y checkpoint .-> G[revision humana]
```

## 4. Preparación del entorno

Importamos LangGraph, un checkpointer en memoria y Pydantic. Las herramientas son locales y el grafo no llama a un modelo generativo.

In [ ]:
import re
from typing import Any, Literal, TypedDict

import pandas as pd
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, ConfigDict, Field

from llm_engineering.environment import preparar_entorno
from llm_engineering.tools import execute_tool

raiz_curso = preparar_entorno()

## 5. Definición del problema

El workflow recibe una solicitud: si contiene una operación explícita, llama a la calculadora segura; de lo contrario, responde directamente. La tarea ilustra control de flujo, no una clasificación semántica general.

## 6. Línea base

Una función directa puede resolver una operación, pero no expresa nodos, persistencia o pausa. Registramos el resultado para comparar con el workflow explícito.

In [ ]:
solicitud_base = 'calcula 4 + 5'
resultado_base = execute_tool('calculadora', {'expression': '4 + 5'})
print(resultado_base.output)

## 7. Implementación

El estado contiene solicitud, ruta, argumentos y salida. Cada nodo devuelve únicamente los campos que actualiza; un modelo Pydantic valida el resultado final y hace explícito el contrato entre workflow y aplicación.

In [ ]:
class Estado(TypedDict, total=False):
    solicitud: str
    ruta: Literal['herramienta', 'directa']
    argumentos: dict[str, Any]
    salida_cruda: dict[str, Any]
    salida_validada: dict[str, Any]


class RespuestaWorkflow(BaseModel):
    model_config = ConfigDict(extra='forbid')

    ruta: Literal['herramienta', 'directa']
    estado: Literal['ok', 'error']
    respuesta: str = Field(min_length=1)
    tool_result: dict[str, Any] | None = None


def clasificar(estado: Estado) -> dict[str, Any]:
    """Selecciona ruta herramienta solo ante una operacion aritmetica."""
    expresion = re.search(r'[-+]?\d+(?:\s*[-+*/]\s*[-+]?\d+)+', estado['solicitud'])
    if 'calcula' in estado['solicitud'].casefold() and expresion:
        return {'ruta': 'herramienta', 'argumentos': {'expression': expresion.group(0)}}
    return {'ruta': 'directa'}


def nodo_herramienta(estado: Estado) -> dict[str, Any]:
    """Ejecuta la calculadora y conserva su resultado como estado."""
    resultado = execute_tool('calculadora', estado.get('argumentos', {}))
    return {'salida_cruda': {
        'ruta': 'herramienta',
        'estado': 'ok' if resultado.success else 'error',
        'respuesta': resultado.output,
        'tool_result': resultado.model_dump(),
    }}


def nodo_directo(estado: Estado) -> dict[str, Any]:
    """Responde sin herramienta para solicitudes generales."""
    return {'salida_cruda': {
        'ruta': 'directa',
        'estado': 'ok',
        'respuesta': f"Recibí: {estado['solicitud']}",
        'tool_result': None,
    }}


def validar_salida(estado: Estado) -> dict[str, Any]:
    """Valida el contrato de salida antes de terminar."""
    respuesta = RespuestaWorkflow.model_validate(estado['salida_cruda'])
    return {'salida_validada': respuesta.model_dump(mode='json')}

constructor = StateGraph(Estado)
constructor.add_node('clasificador', clasificar)
constructor.add_node('herramienta', nodo_herramienta)
constructor.add_node('respuesta_directa', nodo_directo)
constructor.add_node('validacion', validar_salida)
constructor.add_edge(START, 'clasificador')
constructor.add_conditional_edges(
    'clasificador',
    lambda estado: estado['ruta'],
    {'herramienta': 'herramienta', 'directa': 'respuesta_directa'},
)
constructor.add_edge('herramienta', 'validacion')
constructor.add_edge('respuesta_directa', 'validacion')
constructor.add_edge('validacion', END)
workflow = constructor.compile()
print('Nodos:', list(constructor.nodes))

## 8. Experimentos

Comparamos una solicitud de herramienta y una directa. La única diferencia es el texto de entrada; inspeccionamos ruta y salida tipada para observar qué nodos se activaron.

In [ ]:
entradas = [
    {'solicitud': 'calcula 4 + 5'},
    {'solicitud': 'hola, necesito ayuda'},
]
filas = []
for entrada in entradas:
    estado_final = workflow.invoke(entrada)
    filas.append({
        'solicitud': entrada['solicitud'],
        'ruta': estado_final['ruta'],
        'estado': estado_final['salida_validada']['estado'],
        'respuesta': estado_final['salida_validada']['respuesta'],
    })
tabla_rutas = pd.DataFrame(filas)
display(tabla_rutas)

## 9. Evaluación

Verificamos que cada ruta produce una salida Pydantic válida y que solo la ruta de herramienta ejecuta una llamada local. La pausa humana se configura antes del nodo de herramienta; el estado persiste con un identificador de hilo.

In [ ]:
assert tabla_rutas['estado'].eq('ok').all()
assert tabla_rutas['ruta'].tolist() == ['herramienta', 'directa']

workflow_pausado = constructor.compile(
    checkpointer=MemorySaver(),
    interrupt_before=['herramienta'],
)
configuracion_hilo = {'configurable': {'thread_id': 'aprobacion-01'}}
estado_pausado = workflow_pausado.invoke(
    {'solicitud': 'calcula 4 + 5'},
    configuracion_hilo,
)
print('Pausa antes de ejecutar; salida cruda presente:', 'salida_cruda' in estado_pausado)

APROBACION_HUMANA_SIMULADA = True
if APROBACION_HUMANA_SIMULADA:
    estado_reanudado = workflow_pausado.invoke(None, configuracion_hilo)
    print('Resultado tras reanudar:', estado_reanudado['salida_validada'])

metricas = pd.DataFrame([{
    'rutas_probadas': len(tabla_rutas),
    'salidas_validas': int(tabla_rutas['estado'].eq('ok').sum()),
    'llamadas_llm': 0,
    'workflow_pausado': 'salida_cruda' not in estado_pausado,
}])
display(metricas)
metricas.to_csv(raiz_curso / 'outputs' / '10_01_graph_basics.csv', index=False)

## 10. Discusión

LangGraph hace explícita la transición de estado y el punto de pausa, pero el checkpointer no implementa por sí solo un proceso humano. La aplicación debe mostrar la acción pendiente, obtener aprobación y reanudar el mismo `thread_id`. No ejecutes herramientas sensibles sin autorización.

## 11. Ejercicios

1. Añade un nodo de registro entre clasificador y ruta final.
2. Introduce una ruta condicional adicional para pedir aclaración.
3. Cambia el estado a una clase Pydantic y compara el acceso con TypedDict.
4. Rechaza la reanudación si la aprobación humana es falsa.

## 12. Desafío

Añade un checkpointer persistente, simula dos ejecuciones con el mismo hilo y demuestra qué estado se conserva. Define además una estrategia de expiración y protección de datos.

## 13. Ideas clave

- El estado es el contrato entre nodos.
- Los bordes expresan secuencia y los condicionales expresan routing.
- Checkpoint y thread ID permiten pausar y reanudar.
- Human-in-the-loop requiere una decisión explícita de la aplicación.